# 公开 API 参数与调用示例

本 Notebook 按数据准备、分箱、编码、筛选、建模、评估、规则、报告和导出组织。
在项目根目录或 `examples/` 运行，使用本地 `hscredit_yyp.xlsx` 的完整 970 行验证数据。
该数据含业务信息，不随包发布；示例不依赖已执行的其他 Notebook。

目标 `FPD=1` 表示坏样本；分数方向必须按业务含义指定。统计、图形和报告均使用库内接口。
输出写入 `outputs/api_reference`，可通过环境变量 `HSCREDIT_EXAMPLE_OUTPUT` 修改。

In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd
import hscredit

root = Path.cwd() if (Path.cwd() / "examples").is_dir() else Path.cwd().parent
data = pd.read_excel(root / "examples" / "hscredit_yyp.xlsx")
features = ["衡枢鉴真分老客版", "近六个月非银多头机构数", "青云24"]
X, y = data[features], data["FPD"]
output = Path(os.environ.get("HSCREDIT_EXAMPLE_OUTPUT", root / "outputs" / "api_reference"))
output.mkdir(parents=True, exist_ok=True)
assert set(y.unique()) == {0, 1}
print(f"验证样本：{len(data)} 行，建模特征：{len(features)} 个")

## 分箱：方法枚举、输出枚举与未知类别

`OptimalBinning.VALID_METHODS` 是当前实现的方法集合。`transform` 的 `metric` 为
`indices`、`bins` 或 `woe`。训练后复用规则，数值切点属于右侧箱。
未知类别采用 `handle_unknown="value"` 时进入 `-3` 箱；`"raise"` 则拒绝未知类别。

In [ ]:
from hscredit.core.binning import OptimalBinning
from hscredit.core.metrics import compute_bin_stats

binner = OptimalBinning(method="quantile", max_n_bins=5, n_jobs=1).fit(X, y)
indices = binner.transform(X, metric="indices")
labels = binner.transform(X, metric="bins")
woe = binner.transform(X, metric="woe")
table = compute_bin_stats(indices[features[0]].to_numpy(), y.to_numpy())
assert int(table["样本总数"].sum()) == len(data)
assert indices.shape == labels.shape == woe.shape == X.shape
category_binner = OptimalBinning(method="quantile", handle_unknown="value", n_jobs=1)
category_binner.fit(data[["商品类别"]], y)
unknown = category_binner.transform(pd.DataFrame({"商品类别": ["文档测试未知类别"]}))
assert unknown.iloc[0, 0] == -3
print("分箱方法：", OptimalBinning.VALID_METHODS)

## 编码和筛选

编码器的未知类别策略使用 `value/error/return_nan`，与分箱器的枚举不同。
筛选器通过 `get_support` 获取保留掩码；`SelectionReportCollector` 可采集已拟合筛选器。

In [ ]:
from hscredit.core.encoders import WOEEncoder
from hscredit.core.selectors import VarianceSelector, SelectionReportCollector

encoder = WOEEncoder(features=["商品类别"], handle_unknown="value", n_jobs=1)
encoded = encoder.fit_transform(data[["商品类别"]], y)
assert len(encoded) == len(data)
selector = VarianceSelector(threshold=0.0, n_jobs=1).fit(woe, y)
selected = selector.transform(woe)
collector = SelectionReportCollector(source=selector)
selection_summary = collector.to_dataframe(kind="summary")
selection_details = collector.to_dataframe(kind="details")
assert len(selector.get_support()) == len(features)
assert not selection_summary.empty and not selection_details.empty

## 建模与评估：在训练集拟合转换器

以下重新划分训练集和验证集，再仅使用训练集拟合分箱器。
预测的坏样本概率使用 `higher_risk`；若评估高分更安全的业务评分，应使用 `higher_safe`。

In [ ]:
from sklearn.model_selection import train_test_split
from hscredit.core.models import LogisticRegression
from hscredit.core.metrics import auc, ks, psi

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.3, stratify=y, random_state=42)
train_binner = OptimalBinning(method="quantile", max_n_bins=5, n_jobs=1).fit(X_train, y_train)
train_woe = train_binner.transform(X_train, metric="woe")
valid_woe = train_binner.transform(X_valid, metric="woe")
model = LogisticRegression(max_iter=200, calculate_stats=False, positive_woe_coef=False)
model.fit(train_woe, y_train)
prob = model.predict_proba(valid_woe)[:, 1]
evaluation = {"AUC": auc(y_valid, prob, score_direction="higher_risk"), "KS": ks(y_valid, prob),
              "PSI": psi(X_train[features[0]], X_valid[features[0]], method="quantile")}
assert all(np.isfinite(value) for value in evaluation.values())
print(evaluation)

## 规则和多标签报告

规则使用 `Rule`；金额统计通过 `amount` 指定。多标签入口使用
`overdue=["MOB1"]`、`dpds=[7, 3, 0]`，默认按 `>` 定义坏样本。
`target` 与 `overdue/dpds` 是两种不同的标签输入方式。

In [ ]:
from hscredit.core.rules import Rule
from hscredit.report import feature_bin_stats, feature_group_binning_summary

rule = Rule("`衡枢鉴真分老客版` < 600")
rule_table = rule.report(data, target="FPD", amount="放款金额")
overdue_table = feature_bin_stats(data, feature=features[0], overdue=["MOB1"], dpds=[7, 3, 0],
                                method="quantile", amount="放款金额", long_format=True, n_jobs=1)
group_tables, group_summary = feature_group_binning_summary(
    data, feature=features[0], methods="quantile", target="FPD", date="放款时间", freq="M", n_jobs=1)
assert not rule_table.empty and not overdue_table.empty and not group_summary.empty

## EDA、金融计算、绘图与 Excel 导出

利率按期传入，支付现金流取负值。`bin_plot(result=True)` 返回 `(图对象, 统计表)`；
其他绘图函数的返回格式以各自文档为准。

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from hscredit.core.eda import data_info
from hscredit.core.financial import pmt, ipmt, ppmt
from hscredit.core.viz import bin_plot
from hscredit.excel import dataframe2excel

overview = data_info(data)
principal = float(data["放款金额"].dropna().iloc[0])
payment = pmt(0.12 / 12, 12, principal, when="end")
assert np.isclose(payment, ipmt(0.12 / 12, 1, 12, principal) + ppmt(0.12 / 12, 1, 12, principal))
figure, plot_table = bin_plot(data, target="FPD", feature=features[0], method="quantile",
                            max_n_bins=5, result=True, save=str(output / "分箱图.png"))
plt.close("all")
end_row, end_col = dataframe2excel(table, str(output / "分箱统计.xlsx"), sheet_name="分箱统计")
assert (output / "分箱图.png").is_file() and (output / "分箱统计.xlsx").is_file()
assert end_row > 2 and end_col > 2
print("数据概况、金融恒等式、图片和 Excel 导出验证通过")

## 完整特征报告

`mode="strict"` 要求必需章节成功；`return_result=True` 返回结构化结果。
只需插入位置时，保留默认值可得到 `(end_row, end_col)`。

In [ ]:
from hscredit.report import auto_feature_analysis

result = auto_feature_analysis(
    data, features=features, target="FPD", date="放款时间", amount="放款金额",
    excel_writer=str(output / "特征分析.xlsx"), output_dir=str(output / "图表"),
    pictures=["bin"], n_jobs=1, mode="strict", return_result=True,
)
assert result.metadata["输入行数"] == len(data)
assert (output / "特征分析.xlsx").is_file()
print("公开 API 示例验证完成")